# MemWeave 2.1
### A Careful Evaluation of Budget-Constrained, Query-Agnostic Memory Retention on LongMemEval-S
**Real-data research notebook (revision of MemWeave 2.0, built from the v2.0 audit)**

MemWeave 2.0 was run end to end. Its own diagnostics showed that the headline "learned algorithm wins" story did not survive fair baselines. This revision keeps the same benchmark and the same no-fabrication rule, and changes the notebook so that **every claim it can print is computed from data and checked against fair baselines**.

## What changed from v2.0, and why

| # | Problem found in v2.0 | Fix in v2.1 |
|---|---|---|
| 1 | The hand-built heuristics (rarity, novelty) were **wrong-signed**: evidence sessions are *less* rare and *less* novel than distractors, so those baselines scored below random. Beating them proved little. | Added sign-corrected baselines (`rarity_signed`, `heuristic_signed`) and one-feature rules (`low_novelty`, `high_centrality`). |
| 2 | A one-line rule ("lowest novelty first") matched the learned network, but it was found by scanning features on all 500 questions, so its score was optimistic. | `single_feature_cv` picks the best feature and direction **inside each training fold only**. This is an honest, non-optimistic "best simple rule". |
| 3 | The MLP was the headline model but was beaten by logistic regression and was unstable across seeds. | Logistic regression is the primary model. The MLP and gradient boosting stay as model-class comparators. This choice is **post-hoc** and is labeled as such. |
| 4 | The review could not explain why single-session question types sit near random. | New diagnostics test a concrete mechanism: evidence sessions for one question are topically similar to **each other**, so a "most similar to another session" rule finds them. |
| 5 | Identifier leakage was detected but never quantified further. | IDs are still never used as features. They are used only to define **evaluation pools** (for example, excluding ShareGPT and UltraChat filler) to test whether the effect survives against same-pipeline distractors. |
| 6 | Nothing prevented the narrative from drifting away from the numbers. | A findings cell computes every verdict (supported / not supported) from the paired statistics. |
| 7 | Pipeline bugs and silent failure modes. | Sanity checks (budget never exceeded, random recall near the budget fraction, every question present once per method), alignment checks between cache and dataset, and a `defaultdict` for retained sets. |

## Status of the evidence

- **Confirmatory (pre-specified in v2.0):** H1 (learned beats recency and random), primary budget (10%), primary metric (evidence recall), paired bootstrap plus Wilcoxon with Holm correction.
- **Exploratory (decided after seeing v2.0 results):** the choice of logistic regression as primary, all fair baselines, the artifact diagnostics, the pool analysis, and H5/H6 below. These are labeled **POST-HOC** in the outputs.
- **Not independent confirmation:** this notebook re-runs with fresh cross-validation seeds on the *same 500 questions*. That removes dependence on the old splits but is not a new dataset. A second dataset is needed before any generality claim.

## 1. Research questions, hypotheses and protocol

**RQ1 (Setting B, primary).** Under a fixed token budget, can a system that sees only the history (not the future question) retain more answer-supporting sessions than recency and random?

**RQ2 (fair baselines).** Does a learned policy beat the best *simple, non-learned* rule? (v2.0 never tested this fairly.)

**RQ3 (mechanism).** Where does the retention signal come from: temporal features, rarity, importance cues, redundancy/cost-aware selection, or pairwise similarity between sessions?

**RQ4 (Setting A, reference).** When the question is known, does a learned scorer beat BM25, TF-IDF and dense retrieval, and do static features add anything beyond query features?

| Hypothesis | Statement | Status |
|---|---|---|
| H1 | Learned retention beats recency and random at matched budgets. | confirmatory |
| H2 | Learned retention beats the fair non-learned rules (sign-corrected heuristic, low-novelty, CV-selected single feature). | post-hoc |
| H3 | Redundancy-aware and cost-aware selection improve over plain top-score filling. | confirmatory (carried from v2.0) |
| H4 | Removing the temporal engine hurts knowledge-update and temporal-reasoning questions most. | confirmatory (carried from v2.0) |
| H5 | The retention advantage is concentrated in questions with 2+ evidence sessions (evidence sessions resemble each other); on single-evidence questions learned policies are near random. | post-hoc |
| H6 | Pairwise-similarity features carry the signal: a similarity-only model matches the full model, and a model without them loses most of its advantage. | post-hoc |

**Fixed in the config cell:** primary budget 10% of each history's tokens (secondary 5/20/30%); primary metric evidence recall; repeated stratified 5-fold CV over 3 fresh seeds; optimizer parameters tuned on an inner validation split of the *training* fold only; paired bootstrap 95% CI over questions, Wilcoxon signed-rank, Holm correction within each comparison family.

**How to run.** Colab or local Jupyter, top to bottom. Feature extraction is the slow step and is cached. If you still have `memweave2_results/feature_cache_memweave-2.0.0_500_*.pkl` from the v2.0 run, put it in that folder and the cache is reused (the feature code is unchanged). New outputs go to `memweave2_results/v2_1/`.

## 1b. Environment and configuration

In [ ]:
import os, sys, json, re, time, math, random, hashlib, platform, pickle, warnings, datetime, itertools
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import rankdata
import sklearn
try:
    from IPython.display import display
except Exception:
    display = print
warnings.filterwarnings("ignore")

CODE_VERSION = "memweave-2.1.0"
FEATURE_VERSION = "memweave-2.0.0"   # feature-extraction code is unchanged from v2.0, so v2.0 feature caches stay valid
SMOKE = os.environ.get("MEMWEAVE_SMOKE") == "1"   # developer pipeline self-test only; leave unset for real runs

CFG = dict(
    seeds=[100, 101, 102],                    # fresh CV split seeds (v2.0 used 0,1,2). Same 500 questions: NOT independent data.
    n_folds=5,
    budget_fracs=[0.05, 0.10, 0.20, 0.30],    # fraction of each history's token count
    primary_budget=0.10,                      # fixed in advance (v2.0 pre-registration)
    k_ret=[3, 5],                             # retrieval cut-offs applied AFTER memory retention (Setting B)
    max_questions=None,                       # set e.g. 100 for a quick trial run
    exact_tokens=True,                        # tiktoken cl100k if available, else whitespace
    use_dense="auto",                         # "auto" = only if a GPU and sentence-transformers are available
    dense_model="sentence-transformers/all-MiniLM-L6-v2",
    dense_chunk_words=120, dense_max_chunks=40,
    topic_sim_percentile=90,                  # supersession topic threshold (per history)
    n_boot=5000,
    run_qa_proxy=False, qa_n=100, qa_model="Qwen/Qwen2.5-1.5B-Instruct", qa_topk=3, qa_words_per_session=900,
    rebuild_cache=False,
)
if SMOKE:
    CFG.update(seeds=[100, 101], n_folds=3, n_boot=400, exact_tokens=False, use_dense=False)

OUT_DIR = Path(os.environ.get("MEMWEAVE_OUT", "memweave2_results")); OUT_DIR.mkdir(exist_ok=True)   # feature cache lives here
RES_DIR = OUT_DIR / "v2_1"; RES_DIR.mkdir(exist_ok=True)                                              # all v2.1 outputs live here
rng_global = np.random.default_rng(12345); random.seed(12345); np.random.seed(12345)

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)
print("Code version:", CODE_VERSION, "| feature version:", FEATURE_VERSION, "| SMOKE self-test mode:", SMOKE)
print(json.dumps(CFG, indent=1, default=str))

## 2. Download the real benchmark

LongMemEval-S cleaned (about 265 MB). Do **not** substitute the oracle variant; it contains only evidence sessions and makes memory selection trivial. If you already have the file, set the `MEMWEAVE_DATA` environment variable or edit `DATA_PATH`.

In [ ]:
DATA_DIR = Path("data/longmemeval"); DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path(os.environ.get("MEMWEAVE_DATA", DATA_DIR / "longmemeval_s_cleaned.json"))
DATA_URL = ("https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/"
            "resolve/main/longmemeval_s_cleaned.json")

if not DATA_PATH.exists():
    print("Downloading LongMemEval-S (several minutes)...")
    try:
        import urllib.request
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    except Exception as exc:
        if DATA_PATH.exists(): DATA_PATH.unlink()
        raise RuntimeError("Download failed. Fetch longmemeval_s_cleaned.json manually from the "
                           "Hugging Face dataset page and set DATA_PATH.") from exc
print("Dataset:", DATA_PATH, "| MB:", round(DATA_PATH.stat().st_size / 1e6, 1))

## 3. Load, audit, and leakage check

Besides the schema audit, this cell checks for **label-leakage channels**. Benchmarks sometimes encode evidence status in identifiers or annotations. Session IDs and the per-turn `has_answer` flag are **never used as features** anywhere in this notebook. IDs are used in exactly two places, both for *evaluation*, never for scoring: this audit, and the evaluation-pool analysis in Section 16.

In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)
if not isinstance(raw, list) or not raw:
    raise ValueError("Expected a non-empty top-level JSON list.")
if CFG["max_questions"]:
    sel = np.random.default_rng(0).permutation(len(raw))[:CFG["max_questions"]]
    raw = [raw[i] for i in sorted(sel)]
required = {"question_id", "question", "answer", "haystack_sessions", "haystack_session_ids", "answer_session_ids"}
missing = required - set(raw[0])
if missing: raise ValueError(f"Unexpected schema; missing: {missing}")
if len({r["question_id"] for r in raw}) != len(raw): raise ValueError("Duplicate question_id values.")
print("Questions:", len(raw), "| keys:", sorted(raw[0].keys()))
print("Has haystack_dates:", "haystack_dates" in raw[0], "| has question_date:", "question_date" in raw[0])

def id_kind(i):
    """Coarse source of a session ID. Used ONLY to define evaluation pools and audit tables, never as a feature."""
    i = str(i)
    if i.startswith("answer_"): return "answer_prefixed"
    if i.startswith("sharegpt"): return "sharegpt"
    if i.startswith("ultrachat"): return "ultrachat"
    return "other_generated"

rows, kind_counts, kind_gold = [], Counter(), Counter()
for r in raw:
    ids = [str(x) for x in r["haystack_session_ids"]]; gold = {str(x) for x in (r.get("answer_session_ids") or [])}
    if len(ids) != len(r["haystack_sessions"]): raise ValueError(f"{r['question_id']}: ids/sessions length mismatch")
    for i in ids:
        kind_counts[id_kind(i)] += 1; kind_gold[id_kind(i)] += int(i in gold)
    rows.append(dict(qid=r["question_id"], qtype=r.get("question_type", "unknown"), n_sessions=len(ids),
                     n_gold=len(gold), missing_gold=len(gold - set(ids)),
                     gold_prefix=np.mean([g.startswith("answer_") for g in gold]) if gold else np.nan,
                     nongold_prefix=np.mean([i.startswith("answer_") for i in ids if i not in gold]) if len(ids) > len(gold) else np.nan))
audit = pd.DataFrame(rows)
display(audit.qtype.value_counts().rename("count").to_frame())
display(audit[["n_sessions", "n_gold", "missing_gold"]].describe())
if audit.missing_gold.sum() > 0: raise ValueError("Some gold session IDs are not in the haystack.")
print("Questions with NO gold evidence (excluded from recall metrics):", int((audit.n_gold == 0).sum()))

print("\nSession sources by ID prefix (audit only):")
display(pd.DataFrame({"sessions": kind_counts, "gold_sessions": kind_gold}).assign(
    gold_share=lambda d: d.gold_sessions / d.sessions).round(4))

gp, ngp = audit.gold_prefix.mean(), audit.nongold_prefix.mean()
print(f"LEAKAGE CHECK  fraction of gold IDs starting with 'answer_': {gp:.3f} | of non-gold IDs: {ngp:.3f}")
ID_LEAK = bool(gp > 0.5 and ngp < 0.05)
if ID_LEAK:
    print("CONFIRMED: session IDs encode evidence status in this file. They are NOT used as features anywhere in this "
          "notebook. State this in the paper, and never add an ID-derived feature.")
has_flag = any(isinstance(t, dict) and "has_answer" in t for s in raw[0]["haystack_sessions"] for t in (s if isinstance(s, list) else []))
print("Per-turn 'has_answer' flags present:", has_flag, "(never used for selection)")

## 4. MemWeave architecture and how it maps to code (unchanged from v2.0)

```
 conversation history (sessions, in time order)
          │
          ▼
 [1] Memory Encoder ────────── session → memory unit (user/assistant text, entities, token cost)
          │
          ├─► [2] Temporal Intelligence Engine ── later-session topic similarity, entity/number conflict
          │                                        → supersession signal, "latest on topic" flag
          ├─► [3] Rarity & Importance Estimator ─ rare-term mass, uniqueness/novelty, first-person,
          │                                        numeric/entity/time-reference/update-marker density
          ▼
 [4] Adaptive Utility Network ─ learns P(session is future evidence) from the features above
          │                      (trained on training-fold evidence links only)
          ▼
 [5] Budget-Constrained Optimizer ─ greedy marginal utility per token^γ with a redundancy penalty λ
          │                          and a supersession penalty ν (all tuned on inner validation)
          ▼
 retained memory ──► [6] Future-Query Retrieval (BM25 over retained memory) ──► evidence recall@k
```

Setting B uses components 1–5 with **no query information**. Setting A adds query-conditioned features (TF-IDF, BM25, optional dense similarity) to the same network.

**Memory unit.** One complete session. This keeps the gold mapping (`answer_session_ids`) exact. Finer, turn-level units are a natural extension but are not evaluated here.

## 5. Memory Encoder, Temporal Intelligence Engine, and Rarity/Importance Estimator

All **static** features below use only the conversation history. None uses the question, the gold labels, the session IDs, or the `has_answer` flags. They are therefore legal inputs for Setting B.

| Group | Features | Purpose |
|---|---|---|
| base | recency, days-before-question, log tokens, relative length | cost and age |
| importance | user-turn ratio, user word fraction, user-turn length, first-person density, number density, entity density, time-reference density, update-marker density | personal, factual content likely to be asked about later |
| rarity | mean in-history term rarity, fraction of terms unique to the session | information hard to recover elsewhere |
| uniqueness | novelty (1 − max similarity to any other session), centrality | redundancy / duplication |
| temporal | max similarity to a later session, to an earlier session, **supersession signal**, latest-on-topic flag | knowledge updates |

Supersession signal for session *i*: topic similarity to its most similar **later** session, multiplied by how much the entity/number sets of the two sessions differ. High topic overlap with changed values suggests *i* may have been updated later.

Each feature is also added as a **within-history percentile**, because selection is a within-history ranking problem.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS as STOP

WORD_RE = re.compile(r"[A-Za-z0-9']+")
NUM_RE = re.compile(r"\b\d+(?:[.,]\d+)?\b")
SENT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
DATE_RE = re.compile(r"(\d{4})[/-](\d{1,2})[/-](\d{1,2})")
FIRST_PERSON = {"i", "my", "me", "mine", "i'm", "i've", "i'd", "i'll", "myself", "we", "our", "ours"}
TEMPORAL = {"yesterday", "today", "tomorrow", "ago", "last", "next", "week", "weeks", "month", "months", "year", "years",
            "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday", "january", "february", "march",
            "april", "may", "june", "july", "august", "september", "october", "november", "december", "recently",
            "since", "until", "tonight", "morning", "evening", "weekend"}
UPDATE = {"now", "anymore", "changed", "switched", "moved", "started", "stopped", "upgraded", "new", "recently",
          "previously", "instead", "finally", "quit", "joined", "left", "bought", "sold", "replaced", "updated", "used"}

# ---- tokenizer (exact if available, otherwise whitespace; recorded in the manifest) ----
TOKENIZER = "whitespace-approx"
def count_tokens(text): return max(1, len(str(text).split()))
if CFG["exact_tokens"]:
    try:
        import tiktoken
        _enc = tiktoken.get_encoding("cl100k_base"); _enc.encode("test")
        def count_tokens(text): return max(1, len(_enc.encode(str(text), disallowed_special=())))
        TOKENIZER = "tiktoken/cl100k_base"
    except Exception as e:
        print("tiktoken unavailable -> whitespace approximation:", type(e).__name__)
print("Token counting:", TOKENIZER)

def parse_date(s):
    m = DATE_RE.search(str(s)) if s is not None else None
    if not m: return None
    try: return datetime.date(int(m.group(1)), int(m.group(2)), int(m.group(3))).toordinal()
    except ValueError: return None

def session_turns(session):
    if isinstance(session, dict):
        session = session.get("turns") or session.get("messages") or session.get("conversation") or []
    turns = []
    for t in session:
        if isinstance(t, dict):
            turns.append((str(t.get("role", t.get("speaker", "unknown"))).lower(), str(t.get("content", t.get("text", "")))))
        else:
            turns.append(("unknown", str(t)))      # NOTE: has_answer / ids are deliberately never read
    return turns

def entities(text):
    out = []
    for sent in SENT_RE.split(text):
        for tok in sent.split()[1:]:
            t = tok.strip(".,;:!?\"'()[]{}")
            if len(t) > 1 and t[0].isupper() and t.lower() not in FIRST_PERSON:
                out.append(t.lower())
    return out

def encode_session(session):                       # [1] Memory Encoder
    turns = session_turns(session)
    user = "\n".join(c for r, c in turns if r == "user")
    full = "\n".join(f"{r}: {c}" for r, c in turns)
    return dict(full=full, user=user, n_turns=len(turns), n_user=sum(r == "user" for r, _ in turns),
                tokens=count_tokens(full), ents=set(NUM_RE.findall(user)) | set(entities(user)))

STATIC_NAMES = ["recency", "days_before_q", "log_tokens", "rel_length",
                "user_turn_ratio", "user_word_frac", "log_user_turn_len", "first_person", "number_density",
                "entity_density", "temporal_density", "update_density",
                "rarity_mean", "rare_frac",
                "novelty", "centrality",
                "max_later_sim", "max_earlier_sim", "update_signal", "is_latest_topic"]
GROUPS = {"base": STATIC_NAMES[0:4], "importance": STATIC_NAMES[4:12], "rarity": STATIC_NAMES[12:14],
          "uniqueness": STATIC_NAMES[14:16], "temporal": STATIC_NAMES[16:20]}
XS_NAMES = STATIC_NAMES + [n + "_pct" for n in STATIC_NAMES]
IDX = {n: i for i, n in enumerate(XS_NAMES)}
def pct(col): return rankdata(col) / len(col)

def static_features(units, dts, qd, S):
    n = len(units); F = {k: np.zeros(n) for k in STATIC_NAMES}
    cost = np.array([u["tokens"] for u in units], float)
    uterms = []
    for u in units:
        ws = [w.lower() for w in WORD_RE.findall(u["user"])]
        uterms.append(set(w for w in ws if w not in STOP and len(w) > 2))
    df = Counter(t for s in uterms for t in s)
    off = S[~np.eye(n, dtype=bool)]
    tau = np.percentile(off, CFG["topic_sim_percentile"]) if n > 2 else 1.0
    for i, u in enumerate(units):
        ws = [w.lower() for w in WORD_RE.findall(u["user"])]; nw = max(len(ws), 1)
        tw = max(len(WORD_RE.findall(u["full"])), 1)
        F["recency"][i] = (i + 1) / n
        F["days_before_q"][i] = math.log1p(max(qd - dts[i], 0)) if (qd is not None and dts[i] is not None) else math.log1p(n - 1 - i)
        F["log_tokens"][i] = math.log1p(cost[i]); F["rel_length"][i] = cost[i] / max(cost.max(), 1)
        F["user_turn_ratio"][i] = u["n_user"] / max(u["n_turns"], 1)
        F["user_word_frac"][i] = len(ws) / tw
        F["log_user_turn_len"][i] = math.log1p(len(ws) / max(u["n_user"], 1))
        c = Counter(ws)
        F["first_person"][i] = sum(c[w] for w in FIRST_PERSON) / nw
        F["number_density"][i] = len(NUM_RE.findall(u["user"])) / nw
        F["entity_density"][i] = len(entities(u["user"])) / nw
        F["temporal_density"][i] = sum(c[w] for w in TEMPORAL) / nw
        F["update_density"][i] = sum(c[w] for w in UPDATE) / nw
        ts = uterms[i]
        if ts:
            F["rarity_mean"][i] = float(np.mean([math.log((1 + n) / (1 + df[t])) for t in ts]))
            F["rare_frac"][i] = float(np.mean([df[t] == 1 for t in ts]))
        # [2] temporal intelligence
        later, earlier = S[i, i + 1:], S[i, :i]
        mx_l = float(later.max()) if len(later) else 0.0
        F["max_later_sim"][i] = mx_l; F["max_earlier_sim"][i] = float(earlier.max()) if len(earlier) else 0.0
        if len(later):
            j = i + 1 + int(later.argmax()); Ei, Ej = u["ents"], units[j]["ents"]
            uni = len(Ei | Ej); jac = len(Ei & Ej) / uni if uni else 1.0
            F["update_signal"][i] = mx_l * (1.0 - jac)
        F["is_latest_topic"][i] = float(mx_l < tau)
    sims = S.copy()
    F["novelty"] = 1.0 - sims.max(1) if n > 1 else np.ones(n)
    F["centrality"] = sims.sum(1) / max(n - 1, 1)
    return F

def bm25_scores(q, docs, k1=1.5, b=0.75):
    qt = {w.lower() for w in WORD_RE.findall(q)} - STOP
    ctr = [Counter(w.lower() for w in WORD_RE.findall(d)) for d in docs]
    dl = np.array([sum(c.values()) for c in ctr], float); avg = max(dl.mean(), 1.0); N = len(docs)
    s = np.zeros(N)
    for t in qt:
        dft = sum(1 for c in ctr if t in c)
        if dft == 0: continue
        idf = math.log(1 + (N - dft + 0.5) / (dft + 0.5))
        for j, c in enumerate(ctr):
            tf = c.get(t, 0)
            if tf: s[j] += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[j] / avg))
    return s

def make_dense_fn():
    mode = CFG["use_dense"]
    if mode in (False, "off"): return None
    try:
        import torch
        from sentence_transformers import SentenceTransformer
        if mode == "auto" and not torch.cuda.is_available():
            print("Dense baseline OFF (no GPU; set use_dense=True to force)."); return None
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        model = SentenceTransformer(CFG["dense_model"], device=dev)
    except Exception as e:
        print("Dense baseline OFF:", type(e).__name__, str(e)[:120]); return None
    size, maxc = CFG["dense_chunk_words"], CFG["dense_max_chunks"]
    def fn(q, docs):
        chunks, owner = [], []
        for j, d in enumerate(docs):
            w = d.split(); cs = [" ".join(w[i:i + size]) for i in range(0, len(w), size)][:maxc] or [""]
            chunks += cs; owner += [j] * len(cs)
        E = model.encode(chunks, batch_size=128, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        qe = model.encode([q], convert_to_numpy=True, normalize_embeddings=True)[0]
        out = np.full(len(docs), -1.0); np.maximum.at(out, owner, E @ qe); return out
    print("Dense baseline ON:", CFG["dense_model"], "on", dev); return fn
DENSE_FN = make_dense_fn()
HAS_DENSE = DENSE_FN is not None
QNAMES = ["tfidf", "bm25"] + (["dense"] if HAS_DENSE else [])

def build_bundle(rec, idx):
    t0 = time.perf_counter()
    ids = [str(x) for x in rec["haystack_session_ids"]]
    units = [encode_session(s) for s in rec["haystack_sessions"]]
    dts = [parse_date(d) for d in (rec.get("haystack_dates") or [None] * len(ids))]
    ok = len(dts) == len(ids) and all(d is not None for d in dts)
    order = np.argsort(dts, kind="stable") if ok else np.arange(len(ids))
    ids = [ids[j] for j in order]; units = [units[j] for j in order]; dts = [dts[j] for j in order] if ok else [None] * len(ids)
    n = len(ids); qd = parse_date(rec.get("question_date"))
    gold_set = {str(x) for x in (rec.get("answer_session_ids") or [])}
    docs = [u["full"] for u in units]
    try:
        vec = TfidfVectorizer(stop_words="english", sublinear_tf=True, max_df=0.9); Xd = vec.fit_transform(docs)
    except ValueError:
        vec = TfidfVectorizer(sublinear_tf=True); Xd = vec.fit_transform(docs)
    S = (Xd @ Xd.T).toarray().astype(np.float32); np.fill_diagonal(S, 0.0)
    F = static_features(units, dts, qd, S)
    Xs = np.column_stack([F[k] for k in STATIC_NAMES] + [pct(F[k]) for k in STATIC_NAMES]).astype(np.float32)
    t_static = (time.perf_counter() - t0) * 1e3
    # query-conditioned features (Setting A and retrieval stage of Setting B)
    t1 = time.perf_counter(); q = str(rec["question"])
    tfidf = (Xd @ vec.transform([q]).T).toarray().ravel()
    bm25 = bm25_scores(q, docs); bm25n = bm25 / (bm25.max() + 1e-9)
    qcols = [tfidf, bm25n]
    if HAS_DENSE: qcols.append(DENSE_FN(q, docs))
    Xq = np.column_stack(qcols + [pct(c) for c in qcols]).astype(np.float32)
    return dict(idx=idx, qid=rec["question_id"], qtype=rec.get("question_type", "unknown"), question=q, answer=str(rec.get("answer", "")),
                n=n, ids=ids, cost=np.array([u["tokens"] for u in units], float), gold=np.array([i in gold_set for i in ids]),
                S=S, Xs=Xs, Xq=Xq, bm25=bm25, supersede=F["update_signal"].copy(),
                t_static_ms=t_static, t_query_ms=(time.perf_counter() - t1) * 1e3)

## 6. Extract features for all questions (cached)

This is the slowest cell. Features are query-independent except the `Xq` block, so each history is processed once and reused by every split, seed, method and ablation. The feature code is **identical to v2.0**, so a v2.0 cache is reused if present (a different tokenizer or dense setting in the file name is reported, not hidden). Set `rebuild_cache=True` after changing any feature code.

Before any result is computed, the cache is checked against the dataset: same questions in the same order, and the same number of gold sessions per question.

In [ ]:
def find_cache():
    tok = TOKENIZER.replace("/", "-")
    exact = OUT_DIR / f"feature_cache_{FEATURE_VERSION}_{len(raw)}_{tok}_{'dense' if HAS_DENSE else 'nodense'}.pkl"
    if exact.exists(): return exact
    others = sorted(OUT_DIR.glob(f"feature_cache_{FEATURE_VERSION}_{len(raw)}_*.pkl"))
    return others[0] if others else exact

CACHE = find_cache()
if CACHE.exists() and not CFG["rebuild_cache"]:
    with open(CACHE, "rb") as f: B = pickle.load(f)
    print("Loaded cached features:", CACHE.name, "|", len(B), "questions")
    if TOKENIZER.replace("/", "-") not in CACHE.name:
        print("NOTE: cache was built with a different tokenizer than the current session; token costs come from the cache.")
    n_qcols = B[0]["Xq"].shape[1] // 2                      # tfidf, bm25 (+ dense if it was enabled when the cache was built)
    HAS_DENSE = (n_qcols == 3); QNAMES = ["tfidf", "bm25"] + (["dense"] if HAS_DENSE else [])
else:
    B, t0 = [], time.time()
    for i, rec in enumerate(raw):
        B.append(build_bundle(rec, i))
        if (i + 1) % 25 == 0 or i + 1 == len(raw):
            print(f"  {i + 1}/{len(raw)} questions | {time.time() - t0:.0f}s elapsed", end="\r")
    CACHE = OUT_DIR / f"feature_cache_{FEATURE_VERSION}_{len(raw)}_{TOKENIZER.replace('/', '-')}_{'dense' if HAS_DENSE else 'nodense'}.pkl"
    with open(CACHE, "wb") as f: pickle.dump(B, f)
    print("\nBuilt and cached:", CACHE.name)

# ---- alignment guards: the cache must describe exactly this dataset ----
if [b["qid"] for b in B] != [r["question_id"] for r in raw]:
    raise ValueError("Cache and dataset disagree on question order/ids. Set rebuild_cache=True.")
for b, r in zip(B, raw):
    if int(b["gold"].sum()) != len(set(map(str, r.get("answer_session_ids") or []))):
        raise ValueError(f"{b['qid']}: gold count in cache differs from dataset. Set rebuild_cache=True.")
print("Cache/dataset alignment OK.")
print("Static feature columns:", len(XS_NAMES), "| query feature columns:", B[0]["Xq"].shape[1], "| query features:", QNAMES)
print("Mean sessions/history:", round(np.mean([b["n"] for b in B]), 1),
      "| mean tokens/history:", int(np.mean([b["cost"].sum() for b in B])),
      "| mean gold/question:", round(np.mean([b["gold"].sum() for b in B]), 2))
print("Mean static-feature time (ms/question):", round(np.mean([b["t_static_ms"] for b in B]), 1),
      "| query-feature time:", round(np.mean([b["t_query_ms"] for b in B]), 1))

# question-level metadata used for grouping (n_gold group is derived from labels, for ANALYSIS only)
QINFO = pd.DataFrame({"qid": [b["qid"] for b in B], "qtype": [b["qtype"] for b in B],
                      "n_gold": [int(b["gold"].sum()) for b in B], "n_sessions": [b["n"] for b in B]}).set_index("qid")
QINFO["evidence_group"] = np.where(QINFO.n_gold >= 2, "2+ evidence sessions", "1 evidence session")
display(QINFO.groupby("evidence_group").size().rename("questions").to_frame())

## 7. Budget-Constrained Optimizer and metrics

Given utilities *u_j* (from any scorer), costs *c_j* and a token budget, the optimizer repeatedly adds the session with the highest

&emsp;&emsp;**max(u_j − λ·max_{s∈S} sim(j, s) − ν·supersede_j, ε) / c_j^γ**

that still fits. λ is the redundancy penalty (an MMR-style term), γ controls cost-awareness (γ = 0 is plain top-score filling, γ = 1 is utility per token), and ν penalizes sessions the temporal engine flags as likely superseded. Setting λ = γ = ν = 0 reproduces the standard "rank and fill" policy used for every non-tuned baseline.

**Metrics** (per question): *evidence recall* (share of gold sessions retained), *evidence coverage* (all gold sessions retained), fraction of budget used, and in Setting B *retrieval recall@k* when BM25 is run over the retained memory only.

Two reminders that matter for interpretation. (1) **Chance level:** on the full histories a random policy retains about the budget fraction *f* of the evidence (checked by the sanity tests below), so recall near *f* means "no signal". Inside restricted pools (Section 16) chance differs and is measured directly by the `random` row. (2) **Budget is in tokens, not sessions:** evidence sessions tend to be longer than distractors, so cost-aware policies pay a price for keeping them.

In [ ]:
def select(b, util, p):
    n, cost, budget = b["n"], b["cost"], p["budget"]
    lam, gam, nu = p.get("lam", 0.0), p.get("gamma", 0.0), p.get("nu", 0.0)
    sup, S = b["supersede"], b["S"]
    if np.min(util) < 0:   # negative utilities would all be clipped to the same floor and silently select by index
        raise ValueError("Utilities must be non-negative. Convert a signed score with rank_util() first.")
    chosen, used = [], 0.0
    maxsim = np.zeros(n); avail = np.ones(n, bool)
    while True:
        gain = np.maximum(util - lam * maxsim - nu * sup, 1e-6)
        ratio = gain / np.power(cost, gam)
        ratio[~avail] = -np.inf; ratio[cost > budget - used] = -np.inf
        j = int(np.argmax(ratio))
        if not np.isfinite(ratio[j]): break
        chosen.append(j); used += cost[j]; avail[j] = False; maxsim = np.maximum(maxsim, S[:, j])
    return chosen, used

RET_COLS = [f"ret_r{k}" for k in CFG["k_ret"]]
def metrics(b, chosen, used):
    gold, ng = b["gold"], int(b["gold"].sum()); n = b["n"]
    sel = np.zeros(n, bool); sel[chosen] = True
    out = dict(used_frac=used / b["cost"].sum())
    if ng == 0:
        out.update(recall=np.nan, coverage=np.nan, **{c: np.nan for c in RET_COLS}); return out
    out["recall"] = float((sel & gold).sum() / ng); out["coverage"] = float(out["recall"] == 1.0)
    order = sorted(chosen, key=lambda j: -b["bm25"][j])          # retrieval over retained memory only
    for k, c in zip(CFG["k_ret"], RET_COLS): out[c] = float(gold[order[:k]].sum() / ng) if order else 0.0
    return out

def eval_method(bundles, utils, p, frac):
    rows, kept = [], []
    for b, u in zip(bundles, utils):
        t = time.perf_counter(); chosen, used = select(b, u, {**p, "budget": frac * b["cost"].sum()})
        ms = (time.perf_counter() - t) * 1e3
        rows.append(dict(qid=b["qid"], qtype=b["qtype"], select_ms=ms, **metrics(b, chosen, used))); kept.append(chosen)
    return rows, kept

def combos(grid):
    return [dict(lam=l, gamma=g, nu=n) for l, g, n in itertools.product(grid["lam"], grid["gamma"], grid["nu"])]

def tune_params(val_b, val_u, grid, frac):
    best, best_s = None, -1.0
    for p in combos(grid):                                       # first (simplest) combo wins ties
        r, _ = eval_method(val_b, val_u, p, frac)
        s = np.nanmean([x["recall"] for x in r])
        if s > best_s + 1e-12: best, best_s = p, s
    return best

GRID_FULL = dict(lam=[0.0, 0.15, 0.4], gamma=[0.0, 0.5, 1.0], nu=[0.0, 0.2])
GRID_NONE = dict(lam=[0.0], gamma=[0.0], nu=[0.0])
print("Optimizer grid size (full):", len(combos(GRID_FULL)))
def rank_util(score):
    """Within-history rank transform to (0, 1]. Makes any signed score a valid non-negative utility
    (ties share an average rank). Rankings are unchanged, so rule baselines are exactly 'sort by score'."""
    score = np.asarray(score, float)
    return rankdata(score) / len(score)

## 8. Benchmark-artifact diagnostics (run before any model)

These diagnostics use **only features and gold labels, no model**. They answer: *is the evidence statistically distinguishable from distractors before any learning happens, and why?*

- **Within-history AUC** of each static feature. For each question, this is the probability that a randomly chosen evidence session has a higher feature value than a randomly chosen distractor *from the same history*. 0.5 means no signal; below 0.5 means evidence sessions have **lower** values. CIs are bootstrap over questions.
- **Split by number of evidence sessions.** If evidence sessions for one question resemble *each other* (a plausible benchmark-construction effect), the similarity features should work on questions with 2+ evidence sessions and fail on questions with a single one.
- **Nearest-neighbour test.** For questions with 2+ evidence sessions: how often is an evidence session's most similar session another evidence session, compared with the chance rate?

Everything here is **POST-HOC / exploratory**.

In [ ]:
def within_hist_auc(score, gold):
    g = np.asarray(gold, bool); ng = int(g.sum()); nn = len(g) - ng
    if ng == 0 or nn == 0: return np.nan
    r = rankdata(score)
    return float((r[g].sum() - ng * (ng + 1) / 2) / (ng * nn))

def boot_mean_ci(x, n_boot=None, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0: return np.nan, np.nan, np.nan
    n_boot = n_boot or CFG["n_boot"]; rg = np.random.default_rng(seed)
    bm = rg.choice(x, size=(n_boot, len(x))).mean(1)
    return float(x.mean()), float(np.quantile(bm, .025)), float(np.quantile(bm, .975))

feat_auc = {n: np.array([within_hist_auc(b["Xs"][:, j], b["gold"]) for b in B]) for j, n in enumerate(STATIC_NAMES)}
multi = (QINFO.n_gold.values >= 2)
rows = []
for n, a in feat_auc.items():
    m, lo, hi = boot_mean_ci(a); m1 = np.nanmean(a[~multi]); m2 = np.nanmean(a[multi])
    rows.append(dict(feature=n, auc=m, ci_low=lo, ci_high=hi, auc_1_evidence=m1, auc_2plus_evidence=m2,
                     direction="evidence LOWER" if hi < 0.5 else ("evidence HIGHER" if lo > 0.5 else "no reliable signal")))
FEAT_AUC = pd.DataFrame(rows).sort_values("auc").reset_index(drop=True)
print("Within-history AUC per static feature (0.5 = no signal). POST-HOC diagnostic.")
display(FEAT_AUC.round(3)); FEAT_AUC.to_csv(RES_DIR / "table_feature_auc.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 6)); y = np.arange(len(FEAT_AUC))
ax.barh(y - 0.2, FEAT_AUC.auc_1_evidence, height=0.4, label="questions with 1 evidence session")
ax.barh(y + 0.2, FEAT_AUC.auc_2plus_evidence, height=0.4, label="questions with 2+ evidence sessions")
ax.axvline(0.5, color="k", lw=1); ax.set_yticks(y); ax.set_yticklabels(FEAT_AUC.feature); ax.set_xlabel("within-history AUC")
ax.set_title("Which features separate evidence from distractors?"); ax.legend(fontsize=8); plt.tight_layout()
plt.savefig(RES_DIR / "fig_feature_auc.png", dpi=160); plt.show()

# ---- nearest-neighbour test: is an evidence session's most similar session another evidence session? ----
nn_rows = []
for b in B:
    g = b["gold"]; gi = np.where(g)[0]; n = b["n"]
    if len(gi) < 2 or n < 3: continue
    nn_idx = b["S"][gi].argmax(1)                                      # each evidence session's most similar session
    nn_rows.append(dict(qid=b["qid"], observed=float(np.mean(g[nn_idx])), chance=(len(gi) - 1) / (n - 1),
                        gold_to_gold=float(b["S"][np.ix_(gi, gi)].max(1).mean()),
                        gold_to_nongold=float(b["S"][np.ix_(gi, np.where(~g)[0])].max(1).mean())))
NN = pd.DataFrame(nn_rows)
print(f"\nQuestions with 2+ evidence sessions: {len(NN)}")
print(f"P(most-similar session of an evidence session is another evidence session): observed {NN.observed.mean():.3f} vs chance {NN.chance.mean():.3f}")
print(f"Mean max-similarity: evidence->evidence {NN.gold_to_gold.mean():.3f} | evidence->distractor {NN.gold_to_nongold.mean():.3f}")
NN.to_csv(RES_DIR / "table_nearest_neighbour.csv", index=False)

## 9. Utility models and ablation variants

The utility model predicts whether a session will be gold evidence for the (unseen) question, trained **only on training-fold evidence links**. Positives are rare (about 4% of sessions), so only the *ranking* of utilities matters.

**Primary model: logistic regression** (`MemWeave-LR`). This choice is **POST-HOC**: v2.0 pre-registered the MLP, and the v2.0 run then showed logistic regression and gradient boosting beat it with far lower seed variance. The MLP (`MemWeave-MLP`, the v2.0 model) and gradient boosting stay in as model-class comparators, so the switch is visible rather than hidden.

Ablations are run on the primary model: drop one feature group, switch off one optimizer term, or (new) drop or isolate the **pairwise-similarity features** (`novelty`, `centrality`, `max_later_sim`, `max_earlier_sim`, `update_signal`, `is_latest_topic`), which are the features that compare a session with the other sessions in the same history.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split

class ConstUtility:                                              # fallback if a fold has a single label class
    def __init__(self, v): self.v = v
    def predict_proba(self, X): return np.column_stack([1 - np.full(len(X), self.v), np.full(len(X), self.v)])

def fit_utility(kind, X, y, seed):
    if len(np.unique(y)) < 2: return ConstUtility(float(y.mean()))
    if kind == "logreg":
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5, class_weight="balanced"))
    elif kind == "hgb":
        m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, class_weight="balanced", random_state=seed)
    elif kind == "mlp":
        pos = np.where(y == 1)[0]; rep = int(min(10, max(1, (len(y) - len(pos)) // max(len(pos), 1))))
        idx = np.concatenate([np.arange(len(y))] + [pos] * (rep - 1)); X, y = X[idx], y[idx]
        m = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True,
                         validation_fraction=0.1, n_iter_no_change=10, max_iter=300, random_state=seed))
    else:
        raise ValueError(kind)
    return m.fit(X, y)

SIM_FEATS = ["novelty", "centrality", "max_later_sim", "max_earlier_sim", "update_signal", "is_latest_topic"]

def cols_for(drop_groups=(), drop_feats=(), only_feats=None):
    names = [n for g, ns in GROUPS.items() if g not in drop_groups for n in ns]
    if only_feats is not None: names = [n for n in names if n in only_feats]
    names = [n for n in names if n not in drop_feats]
    return [IDX[n] for n in names] + [IDX[n + "_pct"] for n in names]

def feat_matrix(bundles, cols, with_q):
    X = np.vstack([np.hstack([b["Xs"][:, cols]] + ([b["Xq"]] if with_q else [])) for b in bundles])
    y = np.concatenate([b["gold"] for b in bundles]).astype(int)
    return X, y

def util_list(model, bundles, cols, with_q):
    out = []
    for b in bundles:
        X = np.hstack([b["Xs"][:, cols]] + ([b["Xq"]] if with_q else []))
        out.append(model.predict_proba(X)[:, 1])
    return out

PRIMARY = "MemWeave-LR (primary)"
MLP_NAME, HGB_NAME = "MemWeave-MLP (v2.0 model)", "MemWeave-HGB"
def V(name, kind="logreg", group="ablation", grid=None, drop_groups=(), drop_feats=(), only_feats=None):
    return dict(name=name, kind=kind, group=group, grid=grid or GRID_FULL, drop_groups=drop_groups, drop_feats=drop_feats, only_feats=only_feats)

VARIANTS = [
    V(PRIMARY, group="main"),
    V(MLP_NAME, kind="mlp", group="model_class"),
    V(HGB_NAME, kind="hgb", group="model_class"),
    V("- importance cues", drop_groups=("importance",)),
    V("- rarity", drop_groups=("rarity",)),
    V("- uniqueness/novelty", drop_groups=("uniqueness",)),
    V("- temporal engine", drop_groups=("temporal",), grid=dict(GRID_FULL, nu=[0.0])),
    V("- base (recency/length)", drop_groups=("base",)),
    V("- redundancy penalty", grid=dict(GRID_FULL, lam=[0.0])),
    V("- cost-awareness", grid=dict(GRID_FULL, gamma=[0.0])),
    V("- all pairwise-similarity features", drop_feats=SIM_FEATS, grid=dict(GRID_FULL, nu=[0.0])),
    V("pairwise-similarity features only", only_feats=SIM_FEATS, group="control", grid=dict(GRID_FULL, nu=[0.0])),
]
print("Utility variants:", [v["name"] for v in VARIANTS])

## 10. Baselines

**Setting B (query-agnostic)**, nothing here sees the question:

*v2.0 baselines (kept for continuity)*
- `recency`, `random` (seeded), `user_density` (first-person density), `rarity_only`, `heuristic_fill` (equal-weight first-person + rarity + novelty), `heuristic_mmr`. Two of these inputs (rarity and novelty) point the **wrong way** on this benchmark, which Section 8 shows; they are kept only so the v2.0 comparison is reproducible.

*Fair baselines (new, POST-HOC)*, no learning:
- `rarity_signed`: lowest rarity first. `heuristic_signed`: the same heuristic with rarity and novelty flipped.
- `low_novelty`: most similar to another session first. `high_centrality`: highest mean similarity to the other sessions first.
- **`single_feature_cv`**: the single best (feature, direction) pair chosen by recall at the primary budget **using the training questions of each fold only**. This replaces the optimistic scan in the v2.0 review with an honest, nested estimate of "the best one-line rule".

**Setting A (query-conditioned)**, the question is known:
- `recency`, `tfidf`, `bm25`, `dense` (if available), `bm25_mmr`, and the learned scorers: `MemWeave-LR-Q` (static + query features), `MemWeave-LR-Q query-only` (**control**: do static features add anything?), and `MemWeave-MLP-Q` (the v2.0 model).

All non-tuned baselines use plain rank-and-fill (λ = γ = ν = 0). Rule baselines are converted with `rank_util`, which preserves their ranking exactly and keeps utilities non-negative.

In [ ]:
HEUR_COLS = ["first_person_pct", "rarity_mean_pct", "novelty_pct"]
def col(name): return lambda b: b["Xs"][:, IDX[name]]
def qcol(name): return lambda b: b["Xq"][:, QNAMES.index(name)]
def heuristic(b): return np.mean([b["Xs"][:, IDX[c]] for c in HEUR_COLS], axis=0)
def heuristic_signed(b):
    return np.mean([b["Xs"][:, IDX["first_person_pct"]], 1 - b["Xs"][:, IDX["rarity_mean_pct"]], 1 - b["Xs"][:, IDX["novelty_pct"]]], axis=0)
def signed(name, sgn): return lambda b: rank_util(sgn * b["Xs"][:, IDX[name]])

def pick_single_feature(train_b, frac):
    """Best (feature, direction) by mean recall at `frac` over the given TRAINING bundles only."""
    best = (None, 1, -1.0)
    for n in STATIC_NAMES:
        for sgn in (1, -1):
            u = [rank_util(sgn * b["Xs"][:, IDX[n]]) for b in train_b]
            r, _ = eval_method(train_b, u, {}, frac)
            s = float(np.nanmean([x["recall"] for x in r]))
            if s > best[2] + 1e-12: best = (n, sgn, s)
    return best

RETAINED = defaultdict(dict)
def run_fold(seed, fold, tr, te):
    types = [B[i]["qtype"] for i in tr]
    try: tr_in, tr_val = train_test_split(tr, test_size=0.2, random_state=seed, stratify=types)
    except ValueError: tr_in, tr_val = train_test_split(tr, test_size=0.2, random_state=seed)
    BIN, BVAL, BTR, BTE = [[B[i] for i in s] for s in (tr_in, tr_val, tr, te)]
    rng = np.random.default_rng(1000 * seed + fold)
    rows, PF = [], CFG["primary_budget"]

    def record(setting, method, group, utils_val, utils_te, grid, kind="-"):
        p = tune_params(BVAL, utils_val, grid, PF) if len(combos(grid)) > 1 else combos(grid)[0]
        for frac in CFG["budget_fracs"]:
            r, kept = eval_method(BTE, utils_te, p, frac)
            for x, b, ch in zip(r, BTE, kept):
                x.update(seed=seed, fold=fold, setting=setting, method=method, group=group, kind=kind, budget=frac,
                         params=json.dumps({k: v for k, v in p.items()}))
                rows.append(x)
                if abs(frac - PF) < 1e-9 and seed == CFG["seeds"][0]: RETAINED[(setting, method)][b["idx"]] = ch

    def simple(setting, method, fn, grid=GRID_NONE, group="baseline"):
        record(setting, method, group, [fn(b) for b in BVAL], [fn(b) for b in BTE], grid)

    # ---------------- Setting B: query-agnostic retention ----------------
    simple("B", "recency", col("recency"))
    uv = [rng.random(b["n"]) for b in BVAL]; ut = [rng.random(b["n"]) for b in BTE]
    record("B", "random", "baseline", uv, ut, GRID_NONE)
    simple("B", "user_density", col("first_person_pct"))
    simple("B", "rarity_only", col("rarity_mean_pct"))
    simple("B", "heuristic_fill", heuristic)
    simple("B", "heuristic_mmr", heuristic, grid=dict(lam=[0.0, 0.15, 0.4], gamma=[0.0, 0.5, 1.0], nu=[0.0]))
    # fair, non-learned baselines (POST-HOC)
    simple("B", "rarity_signed", signed("rarity_mean", -1), group="fair_baseline")
    simple("B", "heuristic_signed", heuristic_signed, group="fair_baseline")
    simple("B", "low_novelty", signed("novelty", -1), group="fair_baseline")
    simple("B", "high_centrality", signed("centrality", +1), group="fair_baseline")
    fname, fsgn, _ = pick_single_feature(BTR, PF)                    # selected on training questions only
    fn = signed(fname, fsgn)
    record("B", "single_feature_cv", "fair_baseline", [fn(b) for b in BVAL], [fn(b) for b in BTE], GRID_NONE, kind=f"{fname}{'+' if fsgn > 0 else '-'}")
    # learned variants
    for v in VARIANTS:
        cols = cols_for(v["drop_groups"], v["drop_feats"], v["only_feats"])
        X, y = feat_matrix(BIN, cols, with_q=False)
        m = fit_utility(v["kind"], X, y, seed)
        record("B", v["name"], v["group"], util_list(m, BVAL, cols, False), util_list(m, BTE, cols, False), v["grid"], v["kind"])

    # ---------------- Setting A: query-conditioned ----------------
    simple("A", "recency", col("recency"))
    simple("A", "tfidf", qcol("tfidf"))
    simple("A", "bm25", qcol("bm25"))
    if HAS_DENSE: simple("A", "dense", lambda b: rank_util(qcol("dense")(b)))   # cosine can be negative; rank_util keeps the ranking, makes it a valid utility
    simple("A", "bm25_mmr", qcol("bm25"), grid=dict(lam=[0.0, 0.15, 0.4, 0.8], gamma=[0.0], nu=[0.0]))
    for name, kind, with_static, group in [(A_PRIMARY, "logreg", True, "main"), (A_QONLY, "logreg", False, "control"),
                                          (A_MLP, "mlp", True, "model_class")]:
        cols = cols_for(()) if with_static else []
        X, y = feat_matrix(BIN, cols, with_q=True)
        m = fit_utility(kind, X, y, seed)
        record("A", name, group, util_list(m, BVAL, cols, True), util_list(m, BTE, cols, True), GRID_FULL, kind)
    return rows

A_PRIMARY, A_QONLY, A_MLP = "MemWeave-LR-Q", "MemWeave-LR-Q query-only", "MemWeave-MLP-Q (v2.0 model)"

## 11. Run the repeated cross-validation, then sanity-check it

Repeated stratified 5-fold CV over the configured seeds. Every question is a test question exactly once per seed; the utility models, the single-feature selection and all optimizer parameters are fit or tuned **without** that fold's labels.

Immediately afterwards a set of sanity checks runs. They are cheap and catch silent failures: a budget being exceeded, a random baseline that is not near chance, or a question missing from some method.

In [ ]:
ALL, t0 = [], time.time()
idx_all = np.arange(len(B)); types_all = [b["qtype"] for b in B]
for seed in CFG["seeds"]:
    skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(idx_all, types_all)):
        ALL += run_fold(seed, fold, tr, te)
        print(f"seed {seed} fold {fold + 1}/{CFG['n_folds']} done | {time.time() - t0:.0f}s", end="\r")
R = pd.DataFrame(ALL)
print("\nResult rows:", len(R), "| methods:", R.method.nunique())
R.to_csv(RES_DIR / "all_results_raw.csv", index=False)

# ---------------- sanity checks ----------------
problems = []
over = R[R.used_frac > R.budget + 1e-9]
if len(over): problems.append(f"{len(over)} rows use more than their token budget")
n_seeds = len(CFG["seeds"]); n_q = len(B)
cnt = R.groupby(["setting", "method", "budget"]).qid.nunique()
if (cnt != n_q).any(): problems.append("some (setting, method, budget) combinations do not cover every question")
per_seed = R.groupby(["setting", "method", "budget", "qid"]).seed.nunique()
if (per_seed != n_seeds).any(): problems.append("some questions are not evaluated once per seed")
rb = R[(R.method == "random") & (R.setting == "B")].groupby("budget").recall.mean()
for fr, v in rb.items():
    if abs(v - fr) > 0.03: problems.append(f"random baseline recall {v:.3f} is not near the budget fraction {fr:.2f}")
if R.recall.isna().any(): problems.append("NaN recall present although every question should have gold evidence")
print("SANITY CHECKS:", "all passed" if not problems else "PROBLEMS FOUND")
for p in problems: print("  -", p)
print("Random-policy recall by budget (chance level):", {f"{k:.0%}": round(v, 3) for k, v in rb.items()})
sf = R[R.method == "single_feature_cv"].groupby(["seed", "fold"]).kind.first()
print("\nFeature chosen by single_feature_cv in each training fold (+ = higher first, - = lower first):")
display(sf.value_counts().rename("folds").to_frame())

## 12. Main results

Per-question values are first **averaged over seeds** (so each question counts once), then averaged over questions. `seed_std` is the standard deviation across seeds of the macro-average (a direct measure of seed sensitivity). All methods are shown, including the v2.0 baselines that point the wrong way, so you can see which comparisons are meaningful. **Chance level is the budget fraction.**

In [ ]:
def qmeans(d, cols):
    return d.groupby(["method", "qid"])[cols].mean().reset_index()

def main_table(setting, frac, groups=None):
    d = R[(R.setting == setting) & np.isclose(R.budget, frac)]
    if groups: d = d[d.group.isin(groups)]
    cols = ["recall", "coverage", "used_frac"] + (RET_COLS if setting == "B" else [])
    tab = qmeans(d, cols).groupby("method")[cols].mean()
    sd = d.groupby(["method", "seed"]).recall.mean().groupby("method").std().rename("seed_std")
    tab = tab.join(sd).join(d.groupby("method").select_ms.mean().rename("select_ms")).join(d.groupby("method").group.first())
    return tab.sort_values("recall", ascending=False)

PF = CFG["primary_budget"]
print(f"SETTING B (query-agnostic retention), primary budget {PF:.0%} of history tokens. Chance level = {PF:.2f}")
tabB = main_table("B", PF); display(tabB.round(4))
print(f"\nSETTING A (query-conditioned selection), budget {PF:.0%}")
tabA = main_table("A", PF); display(tabA.round(4))
tabB.to_csv(RES_DIR / "table_main_B.csv"); tabA.to_csv(RES_DIR / "table_main_A.csv")

In [ ]:
SHOW_B = ["recency", "random", "heuristic_fill", "low_novelty", "single_feature_cv", "heuristic_signed", PRIMARY, MLP_NAME]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, (setting, show, title) in zip(axes, [("B", SHOW_B, "Setting B: query-agnostic retention"),
                                            ("A", None, "Setting A: query-conditioned selection")]):
    d = R[(R.setting == setting)]
    if show: d = d[d.method.isin(show)]
    for m, g in d.groupby("method"):
        c = g.groupby(["budget", "qid"]).recall.mean().groupby("budget").agg(["mean", "sem"])
        ax.errorbar(c.index * 100, c["mean"], yerr=1.96 * c["sem"], marker="o", capsize=2, label=m,
                    lw=2.4 if m in (PRIMARY, A_PRIMARY) else 1.2)
    ax.plot([5, 30], [0.05, 0.30], "k:", lw=1, label="chance (= budget)")
    ax.set_xlabel("memory budget (% of history tokens)"); ax.set_ylabel("evidence recall"); ax.set_title(title)
    ax.set_ylim(0, 1); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(RES_DIR / "fig_recall_vs_budget.png", dpi=160); plt.show()

## 13. Where does the signal come from? Category and evidence-count analysis

Aggregate scores can hide where a method works. Two breakdowns at the primary budget:

1. **By question type**: the benchmark's six categories (H4 lives here).
2. **By number of evidence sessions** (H5): questions with a single evidence session versus questions with 2+. If a method only works when evidence sessions can find *each other*, it will be near chance on the first group. The grouping uses gold labels, so it is an **analysis** variable only; no method sees it.

In [ ]:
CMP_B = ["random", "recency", "low_novelty", "single_feature_cv", "heuristic_signed", PRIMARY, MLP_NAME]
def group_table(setting, frac, methods, by):
    d = R[(R.setting == setting) & np.isclose(R.budget, frac) & R.method.isin(methods)]
    q = d.groupby(["method", "qid"]).recall.mean().reset_index().join(QINFO[[by]], on="qid")
    t = q.pivot_table(index=by, columns="method", values="recall", aggfunc="mean")[methods]
    t.loc["(all questions)"] = q.groupby("method").recall.mean()[methods]
    t.insert(0, "n_questions", pd.concat([QINFO.groupby(by).size(), pd.Series({"(all questions)": len(QINFO)})]))
    return t

catB = group_table("B", PF, CMP_B, "qtype"); print(f"Setting B @ {PF:.0%}: evidence recall by question type (chance = {PF:.2f})"); display(catB.round(3))
evB = group_table("B", PF, CMP_B, "evidence_group"); print("\nSetting B: evidence recall by number of evidence sessions (POST-HOC, H5)"); display(evB.round(3))
catB.to_csv(RES_DIR / "table_category_B.csv"); evB.to_csv(RES_DIR / "table_evidence_group_B.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw=dict(width_ratios=[3, 1.6]))
catB.drop(index="(all questions)").drop(columns="n_questions").plot(kind="bar", ax=axes[0]); axes[0].set_title(f"By question type @ {PF:.0%} budget")
evB.drop(index="(all questions)").drop(columns="n_questions").plot(kind="bar", ax=axes[1], legend=False); axes[1].set_title("By number of evidence sessions")
for ax in axes:
    ax.axhline(PF, color="k", ls=":", lw=1); ax.set_ylabel("evidence recall"); ax.set_ylim(0, 1); ax.set_xlabel("")
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
axes[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(RES_DIR / "fig_category_B.png", dpi=160); plt.show()

## 14. Statistical significance (paired, with Holm correction inside each family)

Paired comparisons over the same questions: seed-averaged per-question differences, a percentile bootstrap 95% CI of the mean difference, a two-sided Wilcoxon signed-rank test, and Holm correction **within each comparison family**. A comparison is called reliable only when its CI excludes 0. The Holm-adjusted p-value is reported next to it, and neither is treated as proof of a general claim, because all 500 questions come from one benchmark.

Families: **confirmatory** (learned vs recency and random), **fair baselines** (POST-HOC), **model class** (POST-HOC), and Setting A.

In [ ]:
def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); adj = np.empty_like(ps); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (len(ps) - r) * ps[i]); adj[i] = min(1.0, run)
    return adj

def paired(setting, frac, focus, others, family="", n_boot=None, seed=0):
    n_boot = n_boot or CFG["n_boot"]; rng = np.random.default_rng(seed)
    d = R[(R.setting == setting) & np.isclose(R.budget, frac)]
    wide = d.groupby(["qid", "method"]).recall.mean().unstack()
    rows = []
    for o in others:
        diff = (wide[focus] - wide[o]).dropna().to_numpy()
        boots = rng.choice(diff, size=(n_boot, len(diff))).mean(1); lo, hi = np.quantile(boots, [.025, .975])
        try: p = float(stats.wilcoxon(diff).pvalue) if np.any(diff != 0) else 1.0
        except ValueError: p = 1.0
        rows.append(dict(family=family, setting=setting, budget=frac, comparison=f"{focus} - {o}", n=len(diff), mean_diff=diff.mean(),
                         ci_low=lo, ci_high=hi, wilcoxon_p=p))
    out = pd.DataFrame(rows); out["holm_p"] = holm(out.wilcoxon_p.values)
    out["ci_excludes_0"] = (out.ci_low > 0) | (out.ci_high < 0)
    return out

def paired_by_group(setting, frac, focus, other, by, n_boot=None, seed=0):
    n_boot = n_boot or CFG["n_boot"]; rng = np.random.default_rng(seed)
    d = R[(R.setting == setting) & np.isclose(R.budget, frac)]
    diff = (lambda w: (w[focus] - w[other]).dropna())(d.groupby(["qid", "method"]).recall.mean().unstack())
    grp = QINFO.loc[diff.index, by]; rows = []
    for g in sorted(grp.unique()):
        x = diff[grp == g].to_numpy(); bm = rng.choice(x, size=(n_boot, len(x))).mean(1)
        rows.append(dict(group=g, n=len(x), mean_diff=x.mean(), ci_low=np.quantile(bm, .025), ci_high=np.quantile(bm, .975)))
    out = pd.DataFrame(rows); out["ci_excludes_0"] = (out.ci_low > 0) | (out.ci_high < 0); out.insert(0, "comparison", f"{focus} - {other}")
    return out

FAIR = ["low_novelty", "high_centrality", "single_feature_cv", "heuristic_signed", "rarity_signed", "user_density", "heuristic_fill", "heuristic_mmr"]
basesA = [m for m in ["recency", "tfidf", "bm25", "dense", "bm25_mmr", A_QONLY, A_MLP] if m in set(R[R.setting == "A"].method)]
stat_tables = []
for frac in CFG["budget_fracs"]:
    stat_tables += [paired("B", frac, PRIMARY, ["recency", "random"], family="confirmatory (H1)"),
                    paired("B", frac, PRIMARY, FAIR, family="fair baselines (post-hoc, H2)"),
                    paired("B", frac, PRIMARY, [MLP_NAME, HGB_NAME], family="model class (post-hoc)"),
                    paired("A", frac, A_PRIMARY, basesA, family="Setting A")]
STATS = pd.concat(stat_tables, ignore_index=True); STATS.to_csv(RES_DIR / "table_significance.csv", index=False)
show = ["family", "comparison", "mean_diff", "ci_low", "ci_high", "holm_p", "ci_excludes_0"]
for fam in ["confirmatory (H1)", "fair baselines (post-hoc, H2)", "model class (post-hoc)", "Setting A"]:
    print(f"\n{fam}: primary budget {PF:.0%}  (mean_diff = focus minus comparison; positive favours the focus method)")
    display(STATS[(STATS.family == fam) & np.isclose(STATS.budget, PF)][show].round(4))

## 15. Ablations (H3, H4, H6), efficiency and ceilings

**Ablations** compare each variant to the primary model on the same questions with paired bootstrap CIs. A component is **supported** only if removing it produces a CI that excludes 0 in the harmful direction (`mean_diff` = full minus ablated; positive means the component helps).

- **H3** looks at the redundancy-penalty and cost-awareness rows.
- **H4** is tested per question type with a paired CI, instead of comparing two group means. (In v2.0 the knowledge-update means were exactly equal; per-question differences show that was a coincidence of averages, not an identical output.)
- **H5** asks whether each method beats random within each evidence-count group.
- **H6** compares the primary model with the similarity-only model and the model without similarity features.

**Efficiency** reports tokens actually retained, selection latency and feature-extraction cost. **Ceilings** show BM25 retrieval over the *entire* history with no memory limit, which no budgeted memory should be expected to beat.

In [ ]:
abl_names = [v["name"] for v in VARIANTS if v["group"] in ("ablation", "control")]
ABL = paired("B", PF, PRIMARY, abl_names, family="ablations"); ABL["meaning"] = "primary minus variant (positive = removed component helps)"
tab_abl = main_table("B", PF, groups=["main", "ablation", "control", "model_class"])[["recall", "coverage", "seed_std"]]
print("Ablations (Setting B, primary budget):"); display(tab_abl.round(4))
display(ABL[["comparison", "mean_diff", "ci_low", "ci_high", "holm_p", "ci_excludes_0"]].round(4))
tab_abl.to_csv(RES_DIR / "table_ablation.csv"); ABL.to_csv(RES_DIR / "table_ablation_paired.csv", index=False)

# H4: temporal engine by question type (paired)
H4 = paired_by_group("B", PF, PRIMARY, "- temporal engine", "qtype")
print("\nH4: effect of the temporal engine by question type (paired; positive = engine helps)"); display(H4.round(4)); H4.to_csv(RES_DIR / "table_h4_temporal.csv", index=False)

# H5: advantage over random within evidence-count groups (POST-HOC)
H5 = pd.concat([paired_by_group("B", PF, m, "random", "evidence_group") for m in [PRIMARY, "low_novelty", "single_feature_cv", MLP_NAME]], ignore_index=True)
print("\nH5 (post-hoc): advantage over random by number of evidence sessions"); display(H5.round(4)); H5.to_csv(RES_DIR / "table_h5_evidence_group.csv", index=False)

# efficiency
eff_methods = ["recency", "random", "heuristic_fill", "low_novelty", "single_feature_cv", PRIMARY, MLP_NAME]
eff = R[(R.setting == "B") & np.isclose(R.budget, PF) & R.method.isin(eff_methods)].groupby("method")[["used_frac", "select_ms"]].mean()
eff.loc["(feature extraction, per history)", "select_ms"] = np.mean([b["t_static_ms"] for b in B])
print("\nEfficiency (Setting B, primary budget):"); display(eff.round(4)); eff.to_csv(RES_DIR / "table_efficiency.csv")

# ceilings: full-history retrieval with no memory budget
ceil = []
for b in B:
    ng = int(b["gold"].sum())
    if ng == 0: continue
    order = np.argsort(-b["bm25"], kind="stable")
    ceil.append({f"full_history_bm25_r{k}": b["gold"][order[:k]].sum() / ng for k in CFG["k_ret"]})
ceil = pd.DataFrame(ceil).mean(); print("\nCeiling: BM25 over the FULL history (no memory limit):"); display(ceil.round(4).to_frame("recall")); ceil.to_csv(RES_DIR / "table_ceiling.csv")

## 16. Does the effect survive against same-pipeline distractors? (POST-HOC)

Session IDs reveal that distractors come from different sources: some are generic chat filler (`sharegpt_*`, `ultrachat_*`), others look like sessions generated by the same pipeline as the evidence. A fair retention test should not be passable merely by telling "benchmark-written" sessions from generic filler.

This cell re-evaluates the **non-learned** rules inside restricted candidate pools. Pairwise-similarity features are **recomputed inside the pool**, so a filtered-out session cannot influence the scores. The IDs only decide *which sessions are in the pool*. They are never a score, never a feature, and no model sees them. The budget is 10% of the **pool's** tokens. **Chance level is the `random` column, not 0.10:** the budget is counted in tokens and a fill-with-skip policy favours short sessions, while evidence sessions (about 3,000 tokens) are much longer than ShareGPT/UltraChat filler (about 1,300) and about as long as the other generated sessions. So random recall differs by pool, and the right quantity to read is each rule's **lift over random** inside the same pool.

- **all sessions**: the normal setting.
- **same-pipeline pool**: evidence sessions plus the non-ShareGPT, non-UltraChat distractors (the hardest realistic comparison available in this file).
- **filler-only pool**: evidence sessions plus ShareGPT/UltraChat filler only.

In [ ]:
POOLS = {"all sessions": None,
         "same-pipeline pool (answer_ + other_generated)": {"answer_prefixed", "other_generated"},
         "filler-only pool (answer_ + sharegpt/ultrachat)": {"answer_prefixed", "sharegpt", "ultrachat"}}

def pool_eval(frac, n_rand=10, seed=0):
    rg = np.random.default_rng(seed); rows = []
    for b in B:
        for pname, kinds in POOLS.items():
            m = np.ones(b["n"], bool) if kinds is None else np.array([id_kind(i) in kinds for i in b["ids"]])
            n = int(m.sum()); g = b["gold"][m]
            if g.sum() == 0 or n < 5: continue
            S = b["S"][np.ix_(m, m)]; cost = b["cost"][m]
            sub = dict(n=n, cost=cost, S=S, supersede=np.zeros(n), bm25=b["bm25"][m], gold=g)
            scores = {"low_novelty": rank_util(-(1 - S.max(1))), "high_centrality": rank_util(S.sum(1) / max(n - 1, 1)),
                      "recency": rank_util(np.arange(n))}
            for meth, u in scores.items():
                ch, used = select(sub, u, dict(budget=frac * cost.sum()))
                rows.append(dict(pool=pname, method=meth, qid=b["qid"], recall=metrics(sub, ch, used)["recall"]))
            rr = []
            for _ in range(n_rand):
                ch, used = select(sub, rg.random(n), dict(budget=frac * cost.sum())); rr.append(metrics(sub, ch, used)["recall"])
            rows.append(dict(pool=pname, method="random", qid=b["qid"], recall=float(np.mean(rr))))
    return pd.DataFrame(rows)

PO = pool_eval(PF)
rows = []
for (pool, meth), g in PO.groupby(["pool", "method"]):
    m, lo, hi = boot_mean_ci(g.recall.values); rows.append(dict(pool=pool, method=meth, n_questions=len(g), recall=m, ci_low=lo, ci_high=hi))
POOL_TAB = pd.DataFrame(rows)
POOL_TAB = POOL_TAB.pivot(index="pool", columns="method", values="recall")[["random", "recency", "high_centrality", "low_novelty"]]
POOL_LIFT = POOL_TAB[["recency", "high_centrality", "low_novelty"]].sub(POOL_TAB["random"], axis=0).add_suffix(" minus random")
print(f"Evidence recall of non-learned rules inside restricted pools @ {PF:.0%} of pool tokens. Chance = the 'random' column of each pool. POST-HOC.")
display(POOL_TAB.round(3)); print("\nLift over random within each pool:"); display(POOL_LIFT.round(3))
POOL_TAB.join(POOL_LIFT).to_csv(RES_DIR / "table_pool_robustness.csv")
PO.to_csv(RES_DIR / "pool_robustness_raw.csv", index=False)

# the same rule, split by number of evidence sessions, inside the hardest pool
hard = PO[PO.pool.str.startswith("same-pipeline")].join(QINFO[["evidence_group"]], on="qid")
print("\nSame-pipeline pool, by number of evidence sessions:")
display(hard.groupby(["evidence_group", "method"]).recall.mean().unstack()[["random", "recency", "high_centrality", "low_novelty"]].round(3))

## 17. Findings (generated from the tables above)

This cell turns the paired statistics into plain verdicts **using fixed rules**: a comparison is "reliable" only if its bootstrap 95% CI excludes 0. It writes `findings.txt`. Paste these verdicts, not hand-written ones, into any report, so the text cannot drift from the numbers. Items tagged **[post-hoc]** were decided after the v2.0 results and are exploratory.

In [ ]:
def verdict(lo, hi):
    return "HIGHER (CI excludes 0)" if lo > 0 else ("LOWER (CI excludes 0)" if hi < 0 else "no reliable difference (CI includes 0)")
def supp(r):
    return "SUPPORTED (removal hurts)" if r.ci_low > 0 else ("CONTRADICTED (removal IMPROVES recall)" if r.ci_high < 0 else "NOT supported (no reliable effect)")
def rec(m, setting="B"): return float(tabB.loc[m, "recall"] if setting == "B" else tabA.loc[m, "recall"])
def get(df, cmp_name, **flt):
    x = df[df.comparison == cmp_name]
    for k, v in flt.items(): x = x[x[k] == v]
    return x.iloc[0]
S1 = STATS[np.isclose(STATS.budget, PF)]
lines = []; add = lines.append
add(f"MemWeave 2.1 findings | Setting B at {PF:.0%} budget (chance = {PF:.2f}) | {len(B)} questions | seeds {CFG['seeds']}")
add("="*100)
add("[confirmatory] H1: learned retention vs recency and random")
for o in ["recency", "random"]:
    r = get(S1, f"{PRIMARY} - {o}", family="confirmatory (H1)")
    add(f"  {PRIMARY} {rec(PRIMARY):.3f} vs {o} {rec(o):.3f}: diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {verdict(r.ci_low, r.ci_high)}")
add("[post-hoc] H2: learned retention vs fair non-learned rules")
for o in FAIR:
    r = get(S1, f"{PRIMARY} - {o}", family="fair baselines (post-hoc, H2)")
    add(f"  vs {o:<18} {rec(o):.3f}: diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {verdict(r.ci_low, r.ci_high)}")
fair_best = max(["low_novelty", "high_centrality", "single_feature_cv", "heuristic_signed", "rarity_signed"], key=rec)
wrong = [m for m in ["rarity_only", "heuristic_fill", "heuristic_mmr"] if rec(m) < rec("random")]
add(f"  Best fair non-learned rule: {fair_best} ({rec(fair_best):.3f}). v2.0 heuristics scoring below random: {wrong if wrong else 'none'}.")
r = get(S1, f"{PRIMARY} - {fair_best}", family="fair baselines (post-hoc, H2)")
add(f"  => Learned model vs best fair rule: {verdict(r.ci_low, r.ci_high)}. "
    + ("The learned model adds nothing demonstrable over a simple rule." if r.ci_low <= 0 else "The learned model beats every simple rule tested."))
add("[post-hoc] Model class")
for o in [MLP_NAME, HGB_NAME]:
    r = get(S1, f"{PRIMARY} - {o}", family="model class (post-hoc)"); add(f"  {PRIMARY} vs {o}: diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {verdict(r.ci_low, r.ci_high)}")
add(f"  Seed std of recall: {PRIMARY} {tabB.loc[PRIMARY, 'seed_std']:.3f} | {MLP_NAME} {tabB.loc[MLP_NAME, 'seed_std']:.3f}")
add("[confirmatory] H3: redundancy-aware and cost-aware selection")
for o in ["- redundancy penalty", "- cost-awareness"]:
    r = get(ABL, f"{PRIMARY} - {o}"); add(f"  removing '{o[2:]}': diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {supp(r)}")
add("[confirmatory] H4: temporal engine (paired, by question type; positive = engine helps; per-category CIs are not multiplicity-adjusted)")
for _, r in H4.iterrows(): add(f"  {r.group:<28} n={int(r.n):<4} diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {verdict(r.ci_low, r.ci_high)}")
r = get(ABL, f"{PRIMARY} - - temporal engine"); add(f"  overall: diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {supp(r)}")
add("[post-hoc] H5: advantage over random by number of evidence sessions")
for _, r in H5.iterrows(): add(f"  {r.comparison:<40} {r.group:<22} n={int(r.n):<4} diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {verdict(r.ci_low, r.ci_high)}")
add("[post-hoc] H6: pairwise-similarity features")
for o in ["pairwise-similarity features only", "- all pairwise-similarity features"]:
    r = get(ABL, f"{PRIMARY} - {o}"); add(f"  {PRIMARY} ({rec(PRIMARY):.3f}) vs {o} ({rec(o):.3f}): diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {verdict(r.ci_low, r.ci_high)}")
add("[post-hoc] Evaluation pools (non-learned rules, recall at 10% of pool tokens)")
add("  (chance is each pool's own random column; read the lift over random)")
for pool, r in POOL_TAB.iterrows(): add(f"  {pool:<50} random {r['random']:.3f} | low_novelty {r['low_novelty']:.3f} (lift {r['low_novelty'] - r['random']:+.3f}) | high_centrality {r['high_centrality']:.3f} (lift {r['high_centrality'] - r['random']:+.3f})")
add("Setting A (question known)")
for o in basesA:
    r = get(S1, f"{A_PRIMARY} - {o}", family="Setting A"); add(f"  {A_PRIMARY} {rec(A_PRIMARY, 'A'):.3f} vs {o} {rec(o, 'A'):.3f}: diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {verdict(r.ci_low, r.ci_high)}")
add(f"  Full-history BM25 ceiling recall@5 (no memory limit): {ceil['full_history_bm25_r5']:.3f}")
add(f"ID leak confirmed in this file: {ID_LEAK}. IDs and has_answer flags were never used as features; IDs defined evaluation pools only.")
add("Caveats: single benchmark; same 500 questions as v2.0 (fresh CV seeds are not independent data); retention is not answer accuracy;")
add("the primary model, fair baselines, H5/H6 and the pool analysis were chosen after seeing v2.0 results.")
FINDINGS = "\n".join(lines); print(FINDINGS); (RES_DIR / "findings.txt").write_text(FINDINGS)

In [ ]:
def norm_text(s): return re.sub(r"[^a-z0-9 ]", "", str(s).lower()).strip()

def qa_context(b, chosen, k, words):
    rec = raw[b["idx"]]
    sess = dict(zip([str(x) for x in rec["haystack_session_ids"]], rec["haystack_sessions"]))
    dates = dict(zip([str(x) for x in rec["haystack_session_ids"]], rec.get("haystack_dates") or [""] * len(sess)))
    top = sorted(chosen, key=lambda j: -b["bm25"][j])[:k]
    parts = []
    for j in top:
        sid = b["ids"][j]; txt = "\n".join(f"{r}: {c}" for r, c in session_turns(sess[sid]))
        parts.append(f"[Session date: {dates.get(sid, '')}]\n" + " ".join(txt.split()[:words]))
    return "\n\n".join(parts)

def eligible_for_qa(b):
    return (b["gold"].sum() > 0 and not b["qid"].endswith("_abs") and "preference" not in b["qtype"]
            and len(b["answer"].split()) <= 8)

QA_RESULTS = None
if CFG["run_qa_proxy"]:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(CFG["qa_model"])
    lm = AutoModelForCausalLM.from_pretrained(CFG["qa_model"], torch_dtype=torch.float16, device_map="auto")
    elig = [b for b in B if eligible_for_qa(b)]
    samp = [elig[i] for i in np.random.default_rng(0).permutation(len(elig))[:CFG["qa_n"]]]
    qa_methods = ["recency", "random", "low_novelty", PRIMARY]
    rows = []
    for b in samp:
        full_ch = list(np.argsort(-b["bm25"], kind="stable")[:CFG["qa_topk"]])
        sources = {m: RETAINED[("B", m)].get(b["idx"], []) for m in qa_methods}; sources["full_history_bm25"] = full_ch
        for m, ch in sources.items():
            ctx = qa_context(b, ch, CFG["qa_topk"], CFG["qa_words_per_session"])
            msgs = [{"role": "system", "content": "Answer the question using only the conversation excerpts. Reply with a short answer. If the excerpts do not contain it, reply 'unknown'."},
                    {"role": "user", "content": f"Excerpts:\n{ctx}\n\nQuestion: {b['question']}\nShort answer:"}]
            ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(lm.device)
            with torch.no_grad(): out = lm.generate(ids, max_new_tokens=48, do_sample=False)
            ans = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
            rows.append(dict(qid=b["qid"], method=m, correct=float(norm_text(b["answer"]) in norm_text(ans))))
    QA_RESULTS = pd.DataFrame(rows); QA_RESULTS.to_csv(RES_DIR / "qa_proxy_results.csv", index=False)
    wide = QA_RESULTS.pivot(index="qid", columns="method", values="correct")
    print("QA proxy accuracy (containment):"); display(wide.mean().round(3).to_frame("accuracy"))
    rng_q = np.random.default_rng(0)
    for o in [m for m in wide.columns if m != PRIMARY]:
        diff = (wide[PRIMARY] - wide[o]).to_numpy(); bt = rng_q.choice(diff, size=(CFG["n_boot"], len(diff))).mean(1)
        print(f"{PRIMARY} - {o}: mean {diff.mean():+.3f}, 95% CI [{np.quantile(bt, .025):+.3f}, {np.quantile(bt, .975):+.3f}]")
else:
    print("QA proxy skipped (CFG['run_qa_proxy'] is False).")

## 18. Downstream answer-level proxy (optional, off by default)

Evidence retention is not answer accuracy. This optional cell closes part of the gap: for a sample of questions it takes each method's **retained memory** (v2.1 compares recency, random, the `low_novelty` rule and the primary model), retrieves the top-k sessions with BM25, and asks a small open instruction model to answer. Correctness is scored by normalized containment of the gold answer in the output.

**This is a cheap proxy, not the benchmark's official metric.** The official evaluation uses an LLM judge (see the LongMemEval repository's `evaluate_qa.py`). Use the proxy for relative comparison between memory policies, and run the official judge before making any QA-accuracy claim in a paper. Preference and abstention questions, and answers longer than 8 words, are excluded because containment is meaningless for them. Set `CFG["run_qa_proxy"] = True` and use a GPU. The generation code below was written but not run by the notebook author.

## 19. Save artifacts and run manifest

Everything a reviewer needs to audit a claim: raw per-question rows, aggregate tables (CSV and LaTeX), figures, the generated findings, and a manifest with the dataset hash, configuration, library versions, tokenizer, the cache file used, and which analyses are post-hoc.

In [ ]:
def to_latex(df, name, caption):
    try:
        s = df.round(4).to_latex(caption=caption, label=f"tab:{name}", float_format="%.4f")
        (RES_DIR / f"{name}.tex").write_text(s)
    except Exception as e:
        print("LaTeX export skipped for", name, "-", type(e).__name__)
to_latex(tabB.drop(columns="group"), "table_main_B", "Setting B: query-agnostic retention at the primary budget.")
to_latex(tabA.drop(columns="group"), "table_main_A", "Setting A: query-conditioned selection at the primary budget.")
to_latex(tab_abl, "table_ablation", "Ablations and model classes in Setting B at the primary budget.")
to_latex(catB, "table_category_B", "Evidence recall by question category (Setting B).")
to_latex(evB, "table_evidence_group_B", "Evidence recall by number of evidence sessions (Setting B).")
to_latex(FEAT_AUC.set_index("feature")[["auc", "ci_low", "ci_high", "auc_1_evidence", "auc_2plus_evidence"]], "table_feature_auc", "Within-history AUC of each static feature.")

pkgs = {}
for p in ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "tiktoken", "sentence_transformers", "torch", "transformers"]:
    try: pkgs[p] = __import__(p).__version__
    except Exception: pkgs[p] = None
manifest = dict(project="MemWeave 2.1", code_version=CODE_VERSION, feature_version=FEATURE_VERSION, dataset="LongMemEval-S cleaned",
                dataset_url=DATA_URL, dataset_file=str(DATA_PATH), dataset_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
                n_questions=len(raw), config=CFG, tokenizer=TOKENIZER, feature_cache=CACHE.name, dense_baseline_enabled=HAS_DENSE,
                smoke_test_mode=SMOKE, id_leak_confirmed=ID_LEAK,
                post_hoc=["primary model = logistic regression", "fair baselines", "feature-AUC diagnostics", "H5", "H6", "pool robustness"],
                note="Fresh CV seeds on the same 500 questions: not independent confirmation.",
                timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                python=sys.version, platform=platform.platform(), packages=pkgs)
(RES_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("Saved to", RES_DIR, ":", sorted(p.name for p in RES_DIR.iterdir() if p.suffix in (".csv", ".tex", ".json", ".png", ".txt")))

## 20. Reporting scaffold (fill from `findings.txt` and the saved tables only)

Every `[RESULT]` below must come from a file in `memweave2_results/v2_1/`. Do not write a number you cannot point to a CSV for.

**Suggested framing: an evaluation study with a negative result and a diagnosis**, not a new-algorithm paper. The notebook's own evidence supports that framing; the findings cell says which parts hold.

**Title (example).** What Does a Query-Agnostic Memory Benchmark Reward? A Fair-Baseline Audit of Retention on LongMemEval-S

**Abstract.** (Problem: long-horizon assistants cannot store full histories and must decide what to keep before knowing future questions.) (Method: a learned utility over temporal, rarity, uniqueness and importance features with a redundancy- and cost-aware budget optimizer, compared with recency, random, sign-corrected heuristics and a nested, CV-selected single-feature rule.) (Result: `[RESULT: findings.txt, table_main_B.csv, table_significance.csv]`, stated exactly as the findings cell states it, including that the learned policy does or does not beat the best simple rule.) (Diagnosis: `[RESULT: table_feature_auc.csv, table_evidence_group_B.csv, table_pool_robustness.csv]`.)

**1. Introduction.** Query-agnostic retention versus query-conditioned retrieval; why fair non-learned baselines matter; contributions limited to what the findings support.

**2. Related work** (verify every citation before submission): LongMemEval (Wu et al., ICLR 2025); LoCoMo (Maharana et al., 2024); MemGPT (Packer et al., 2023); Generative Agents (Park et al., 2023); MemoryBank (Zhong et al., 2023); Mem0 and A-MEM (2025); retrieval-augmented generation (Lewis et al., 2020); BM25 (Robertson and Zaragoza, 2009); Maximal Marginal Relevance (Carbonell and Goldstein, 1998).

**3. Problem formulation.** History of sessions H = (s₁ … s_n) with costs c_i, budget B, unknown future question q with gold evidence set G(q) ⊆ H. Choose M ⊆ H with Σ_{i∈M} c_i ≤ B to maximize E_q |M ∩ G(q)| / |G(q)|.

**4. Method and baselines.** Sections 4–10 above.

**5. Protocol.** Sections 1, 7, 11 above. State plainly which choices are confirmatory and which are post-hoc.

**6. Results.** `[RESULT: Sections 12–17]`. Report the diagnostics (Sections 8, 13, 16) as results in their own right.

**7. Threats to validity (state all of these).**
1. *Benchmark construction artifacts.* Evidence sessions are structurally distinguishable from distractors, and evidence sessions for one question resemble each other. Retention scores on this benchmark can therefore overstate real memory-selection ability. Test on a second dataset (for example LoCoMo) before any generality claim.
2. *Identifier leakage.* Gold IDs carry a distinguishing prefix in this file. IDs were never used as features and defined evaluation pools only.
3. *Post-hoc choices.* The primary model, fair baselines, H5, H6 and the pool analysis came after the v2.0 results. Fresh CV seeds on the same questions do not make them confirmatory.
4. *Session-level units.* No fine-grained fact compression is evaluated.
5. *Retention is not answer accuracy.* Run the official LLM-judge evaluation before any QA claim; Section 18 is a proxy.
6. *Single benchmark, single history length.*
7. *Hyper-parameters tuned at one budget* (the primary one) and reused at others.
8. *Supervision.* Labels are evidence-session membership for training-fold questions, a proxy for future utility under the benchmark's question distribution.

**8. Conclusion and limitations.** Report null and negative results exactly as the findings cell prints them.

## 21. MemWeave 2.2: external validation and claim discipline

This section adds a **locked cross-dataset test** on the official LoCoMo `locomo10.json` release. The source model is trained only on LongMemEval-S; optimizer parameters are selected on a held-out LongMemEval validation subset. The LoCoMo conversations are never used for fitting, feature selection, threshold selection, or hyperparameter tuning.

**Unit alignment.** LoCoMo evidence is annotated as dialogue-turn IDs (`dia_id`), whereas MemWeave stores complete sessions. The adapter maps a question's annotated evidence turns to the containing session(s), and the retrieval metric is therefore **session-level evidence recall**, not turn-level recall. Adversarial questions and questions without usable evidence annotations are excluded from this retrieval analysis and counted in the manifest.

**Conversation-level uncertainty.** LoCoMo has only ten conversations. Questions within one conversation are not independent. Report per-conversation results and use a conversation-cluster bootstrap (not a question-IID bootstrap) for confidence intervals. No tuning on LoCoMo is allowed. If any code or configuration is changed after seeing LoCoMo results, this test becomes exploratory and a fresh external dataset is needed.

**Speaker-feature caveat.** LoCoMo is a two-person dialogue, not a user-assistant transcript. For compatibility, speaker A is mapped to `user` and speaker B to `assistant` solely as a deterministic role convention. This is not a claim that the original participants have assistant/user roles. Report this adaptation as a limitation and include a sensitivity run with speaker-role-dependent importance features removed before submission.

**Download** the official `locomo10.json` from the `snap-research/locomo` repository into `data/locomo/locomo10.json`. The notebook skips the external run if the file is absent. It never silently substitutes another dataset or claims validation without writing a LoCoMo results file.


In [ ]:
# ---- Locked external validation: LongMemEval-trained model -> LoCoMo test ----
from dateutil import parser as date_parser

LOCOMO_PATH = Path(os.environ.get("LOCOMO_DATA", "data/locomo/locomo10.json"))
LOCOMO_RESULTS = RES_DIR / "locomo_external_validation.csv"
LOCOMO_SUMMARY = RES_DIR / "locomo_external_summary.csv"

def _locomo_iso_date(value):
    try:
        return date_parser.parse(str(value), fuzzy=True).date().isoformat()
    except Exception:
        return None

def load_locomo_as_session_records(path):
    """Convert official LoCoMo session/turn evidence to MemWeave session-level records."""
    samples = json.loads(Path(path).read_text(encoding="utf-8"))
    records, excluded = [], []
    for sample in samples:
        conv = sample["conversation"]
        speaker_a, speaker_b = str(conv.get("speaker_a", "speaker_a")), str(conv.get("speaker_b", "speaker_b"))
        session_keys = sorted(
            [k for k in conv if re.fullmatch(r"session_\d+", str(k))],
            key=lambda k: int(k.split("_")[1])
        )
        session_ids, sessions, dates = [], [], []
        dia_to_session = {}
        for sk in session_keys:
            sid = f"{sample.get('sample_id', 'sample')}__{sk}"
            date_value = _locomo_iso_date(conv.get(f"{sk}_date_time"))
            turns = []
            for turn in conv[sk] or []:
                did = str(turn.get("dia_id", ""))
                speaker = str(turn.get("speaker", ""))
                # Deterministic compatibility convention; not semantic user/assistant identity.
                role = "user" if speaker == speaker_a else ("assistant" if speaker == speaker_b else "unknown")
                content = str(turn.get("text", ""))
                caption = str(turn.get("blip_caption", "") or "")
                if caption:
                    content = (content + " [Image caption: " + caption + "]").strip()
                turns.append({"role": role, "content": content})
                if did:
                    dia_to_session[did] = sid
            if not turns:
                continue
            session_ids.append(sid)
            sessions.append(turns)
            dates.append(date_value or "2000-01-01")
        for qi, qa in enumerate(sample.get("qa", [])):
            cat = str(qa.get("category", "unknown")).lower()
            if cat in {"5", "adversarial"}:
                excluded.append({"sample_id": sample.get("sample_id"), "qa_index": qi, "reason": "adversarial"})
                continue
            evidence = qa.get("evidence") or []
            if isinstance(evidence, str):
                evidence = [evidence]
            gold_sessions = sorted({dia_to_session[str(e)] for e in evidence if str(e) in dia_to_session})
            if not gold_sessions:
                excluded.append({"sample_id": sample.get("sample_id"), "qa_index": qi, "reason": "no_mappable_evidence"})
                continue
            records.append({
                "question_id": f"{sample.get('sample_id', 'sample')}__q{qi}",
                "question": str(qa.get("question", "")),
                "answer": str(qa.get("answer", "")),
                "question_type": f"locomo_category_{cat}",
                "haystack_sessions": sessions,
                "haystack_session_ids": session_ids,
                "haystack_dates": dates,
                "question_date": dates[-1] if dates else None,
                "answer_session_ids": gold_sessions,
                "conversation_id": str(sample.get("sample_id", "sample")),
                "qa_category": cat,
            })
    return records, pd.DataFrame(excluded)

def _cluster_bootstrap_ci(per_question, value_col="recall", n_boot=10000, seed=2026):
    # Macro-average conversation means; resample the ten conversations as clusters.
    cm = per_question.groupby("conversation_id")[value_col].mean()
    vals = cm.to_numpy()
    rg = np.random.default_rng(seed)
    draws = rg.choice(vals, size=(n_boot, len(vals)), replace=True).mean(axis=1)
    return float(vals.mean()), float(np.quantile(draws, .025)), float(np.quantile(draws, .975))

if LOCOMO_PATH.exists():
    locomo_raw, LOCOMO_EXCLUDED = load_locomo_as_session_records(LOCOMO_PATH)
    if not locomo_raw:
        raise ValueError("LoCoMo file loaded but no QA rows have mappable evidence; inspect the adapter.")
    L = [build_bundle(rec, i) for i, rec in enumerate(locomo_raw)]

    # Tune optimizer only on a LongMemEval validation split; never inspect LoCoMo labels here.
    src_idx = np.arange(len(B))
    src_types = [b["qtype"] for b in B]
    try:
        src_train_idx, src_val_idx = train_test_split(
            src_idx, test_size=.20, random_state=2026, stratify=src_types
        )
    except ValueError:
        src_train_idx, src_val_idx = train_test_split(src_idx, test_size=.20, random_state=2026)
    cols_ext = cols_for(())
    Xsrc, ysrc = feat_matrix([B[i] for i in src_train_idx], cols_ext, with_q=False)
    source_model_for_tuning = fit_utility("logreg", Xsrc, ysrc, seed=2026)
    val_bundles = [B[i] for i in src_val_idx]
    val_utils = util_list(source_model_for_tuning, val_bundles, cols_ext, with_q=False)
    frozen_optimizer = tune_params(val_bundles, val_utils, GRID_FULL, CFG["primary_budget"])

    # Refit the fixed model class on all LongMemEval-S questions.
    Xall, yall = feat_matrix(B, cols_ext, with_q=False)
    source_model = fit_utility("logreg", Xall, yall, seed=2026)
    learned_utils = util_list(source_model, L, cols_ext, with_q=False)

    ext_rows = []
    rng_ext = np.random.default_rng(2026)
    for b, learned_u in zip(L, learned_utils):
        method_utils = {
            "MemWeave-LR trained on LongMemEval": learned_u,
            "recency": rank_util(b["Xs"][:, IDX["recency"]]),
            "low_novelty": rank_util(-b["Xs"][:, IDX["novelty"]]),
        }
        for method, util in method_utils.items():
            # Only the learned policy receives the source-tuned optimizer.
            # Rule baselines use plain rank-and-fill so their ranking is not changed by tuned penalties.
            params = ({**frozen_optimizer, "budget": CFG["primary_budget"] * b["cost"].sum()}
                      if method == "MemWeave-LR trained on LongMemEval"
                      else {"budget": CFG["primary_budget"] * b["cost"].sum()})
            chosen, used = select(b, util, params)
            ext_rows.append({
                "conversation_id": locomo_raw[b["idx"]]["conversation_id"],
                "qid": b["qid"], "qtype": b["qtype"], "method": method,
                "recall": metrics(b, chosen, used)["recall"],
                "used_frac": used / b["cost"].sum(),
            })
        rand_scores = []
        for _ in range(100):
            chosen, used = select(b, rng_ext.random(b["n"]), {"budget": CFG["primary_budget"] * b["cost"].sum()})
            rand_scores.append(metrics(b, chosen, used)["recall"])
        ext_rows.append({
            "conversation_id": locomo_raw[b["idx"]]["conversation_id"],
            "qid": b["qid"], "qtype": b["qtype"], "method": "random (100-run mean)",
            "recall": float(np.mean(rand_scores)), "used_frac": np.nan,
        })
    LOCOMO_EXT = pd.DataFrame(ext_rows)
    LOCOMO_EXT.to_csv(LOCOMO_RESULTS, index=False)
    summary_rows = []
    for method, g in LOCOMO_EXT.groupby("method"):
        mean, lo, hi = _cluster_bootstrap_ci(g)
        summary_rows.append({
            "method": method, "n_questions": g.qid.nunique(),
            "n_conversations": g.conversation_id.nunique(),
            "macro_conversation_recall": mean, "cluster_ci_low": lo, "cluster_ci_high": hi,
        })
    LOCOMO_SUM = pd.DataFrame(summary_rows)
    LOCOMO_SUM.to_csv(LOCOMO_SUMMARY, index=False)
    LOCOMO_EXCLUDED.to_csv(RES_DIR / "locomo_exclusions.csv", index=False)
    (RES_DIR / "locomo_protocol.json").write_text(json.dumps({
        "dataset": "LoCoMo locomo10.json", "role_mapping": "speaker_a=user; speaker_b=assistant (compatibility convention only)",
        "evidence_unit": "session (dialogue evidence dia_id mapped to containing session)",
        "adversarial_excluded": True, "questions_without_mappable_evidence_excluded": True,
        "source_training_dataset": "LongMemEval-S cleaned", "optimizer_tuned_on": "LongMemEval-S 20% validation split only",
        "locomo_used_for_tuning": False, "budget": CFG["primary_budget"],
        "frozen_optimizer": frozen_optimizer, "random_repetitions": 100,
        "uncertainty": "bootstrap over conversation clusters, not IID questions",
        "excluded_rows": len(LOCOMO_EXCLUDED),
    }, indent=2))
    print("LoCoMo external validation completed; conversation-cluster summaries:")
    display(LOCOMO_SUM)
else:
    print(f"LoCoMo external validation NOT RUN: {LOCOMO_PATH} not found. No independent-validation claim is permitted.")


## 22. Interpretation guardrails for the manuscript

- The three CV seeds are repeated partitions of the **same 500 questions**. They estimate split/seed sensitivity; they are not three independent replications.
- The LR primary-model choice, fair baselines, artifact diagnostics, pool robustness, H5/H6, and model-class comparisons remain exploratory because they were selected after inspecting v2.0.
- The current 10% results show the redundancy penalty is harmful (removing it improves recall); cost-awareness and the temporal optimizer term do not show reliable benefits. Do not claim these optimizer components improve performance.
- The primary LR model is not clearly better than the strongest fair simple rule (`low_novelty` / `single_feature_cv`) in the paired question-level comparison. Therefore, do not claim algorithmic superiority or a novel optimizer contribution. Frame the work as a benchmark audit / empirical study unless future pre-registered tests establish otherwise.
- LoCoMo results must be reported only if `locomo_external_validation.csv` and `locomo_external_summary.csv` are created by Section 21. No LoCoMo score is supplied in this release.
- Evidence recall is not answer accuracy. The optional QA proxy is not an official judge evaluation. Do not make answer-accuracy claims until the benchmark's official evaluator has been run and the model outputs, judge outputs, prompts, model identifiers, and costs are archived.
